<a href="https://colab.research.google.com/github/Zishan06/1_UTS_Praktikum_PI/blob/main/1_UTS_Praktikum_PI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import os
import re
from whoosh.index import create_in, exists_in
from whoosh.fields import Schema, TEXT, ID
from whoosh.analysis import StandardAnalyzer

# daftar stopword bahasa Indonesia sederhana karna tidak boleh pakai NLTK/Sastrawi
STOPWORDS = {"yang", "di", "ke", "dari", "pada", "dalam", "untuk", "dan", "atau", "dengan", "ini", "itu", "adalah", "sebagai", "oleh"}

def preprocess_text(text):
    """
    Fungsi untuk membersihkan teks mentah sesuai standar minimum proyek UTS.
    Meliputi: Case folding, Tokenization (simpel), dan Stopword removal.
    """
    # 1. mengubah semua huruf menjadi huruf kecil
    text = text.lower()

    # regex
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # 2. tokenisasi
    tokens = text.split()

    # 3. stopword removal
    clean_tokens = [word for word in tokens if word not in STOPWORDS]

    # penggabungan
    return " ".join(clean_tokens)

def setup_whoosh_index(index_dir="indexdir"):
    """
    Fungsi untuk membuat skema Whoosh dan menyiapkan direktori indeks.
    """
    # buat folder indeks
    if not os.path.exists(index_dir):
        os.mkdir(index_dir)

    # Mendefinisikan skema Whoosh:
    # - path: Untuk menyimpan lokasi asal file (ID, tidak dianalisis teksnya)
    # - source: Untuk menyimpan nama dataset (etd-usk, kompas, dll)
    # - content: Teks yang sudah di-preprocess dan siap dicari
    schema = Schema(
        path=ID(stored=True, unique=True),
        source=TEXT(stored=True),
        content=TEXT(analyzer=StandardAnalyzer(), stored=True)
    )

    # Membuat indeks baru
    ix = create_in(index_dir, schema)
    return ix

def load_and_index_datasets(base_dataset_path="dataset", index_dir="indexdir"):
    """
    Fungsi utama untuk membaca seluruh dataset dari folder, melakukan preprocessing,
    dan menyimpannya ke dalam indeks Whoosh.
    """
    print(f"Memulai proses Load & Index dari folder '{base_dataset_path}'...")

    # Whoosh index
    ix = setup_whoosh_index(index_dir)
    writer = ix.writer()

    # Dataset
    dataset_folders = ["etd-usk", "etd-ugm", "kompas", "tempo", "mojok"]

    doc_count = 0

    # Iterasi melalui setiap folder dataset
    for folder in dataset_folders:
        folder_path = os.path.join(base_dataset_path, folder)

        # Mengecek apakah folder dataset ada
        if not os.path.exists(folder_path):
            print(f"Peringatan: Folder {folder_path} tidak ditemukan. Lewati.")
            continue

        # Membaca setiap file teks dalam folder
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)

            # baca file
            if os.path.isfile(file_path):
                try:
                    # Membaca isi teks
                    with open(file_path, 'r', encoding='utf-8') as file:
                        raw_text = file.read()

                    # Memanggil fungsi preprocessing
                    cleaned_text = preprocess_text(raw_text)

                    # Menambahkan dokumen ke dalam Whoosh Index
                    writer.add_document(
                        path=file_path,
                        source=folder,
                        content=cleaned_text
                    )
                    doc_count += 1
                except Exception as e:
                    print(f"Gagal memproses file {file_path}: {e}")

    # Menyimpan perubahan ke indeks
    writer.commit()
    print(f"Proses Indexing Selesai! Sebanyak {doc_count} dokumen berhasil diindeks ke dalam '{index_dir}'.")
    return ix



In [39]:
def main_menu():
    """
    CLI Interface.
    """
    while True:
        # tampilan
        print("\n=== INFORMATION RETRIEVAL SYSTEM ===")
        print("[1] Load & Index Dataset")
        print("[2] Search Query")
        print("[3] Exit")
        print("====================================")

        pilihan = input("Pilih menu (1/2/3): ")

        if pilihan == '1':
            # fungsi indexing
            load_and_index_datasets(base_dataset_path="dataset", index_dir="indexdir")
        elif pilihan == '2':
            # input user
            query = input("Masukkan query pencarian: ")
            if query.strip():
                search_and_rank(query, index_dir="indexdir")
            else:
                print("Query tidak boleh kosong.")
        elif pilihan == '3':
            print("Keluar dari program. Terima kasih!")
            sys.exit(0)
        else:
            print("Pilihan tidak valid, silakan coba lagi.")

In [40]:
main_menu()


=== INFORMATION RETRIEVAL SYSTEM ===
[1] Load & Index Dataset
[2] Search Query
[3] Exit
Pilih menu (1/2/3): 1
Memulai proses Load & Index dari folder 'dataset'...
Proses Indexing Selesai! Sebanyak 235 dokumen berhasil diindeks ke dalam 'indexdir'.

=== INFORMATION RETRIEVAL SYSTEM ===
[1] Load & Index Dataset
[2] Search Query
[3] Exit
Pilih menu (1/2/3): Load & Index Dataset
Pilihan tidak valid, silakan coba lagi.

=== INFORMATION RETRIEVAL SYSTEM ===
[1] Load & Index Dataset
[2] Search Query
[3] Exit
Pilih menu (1/2/3): 2
Masukkan query pencarian: teknologi informasi

=== HASIL PENCARIAN ===

Peringkat: 1
Sumber: etd-ugm
Lokasi: dataset/etd-ugm/etd_ugm_4.txt
Cosine Similarity: 0.1781
Isi: laman tidak ditemukan perpustakaan arsip universitas gadjah mada tentang ugm arsip ugm portal akademik pusat ti penelitian surel kontak bahasa indonesia bahasa indonesia english universitas gadjah mada perpustakaan arsip toggle navigation layanan bantuan layanan perpustakaan keanggotaan layanan etd 

SystemExit: 0

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [30]:
import os

BASE_DIR = "/content/dataset"
sumber_dataset = ["etd-usk", "etd-ugm", "kompas", "tempo", "mojok"]

for sumber in sumber_dataset:
    folder = os.path.join(BASE_DIR, sumber)
    os.makedirs(folder, exist_ok=True)
    print(f"{sumber}: {folder} | Ada: {os.path.exists(folder)}")

etd-usk: /content/dataset/etd-usk | Ada: True
etd-ugm: /content/dataset/etd-ugm | Ada: True
kompas: /content/dataset/kompas | Ada: True
tempo: /content/dataset/tempo | Ada: True
mojok: /content/dataset/mojok | Ada: True


In [31]:
import os

for akar, folder, file in os.walk("/content/dataset"):
    print(akar, file)

/content/dataset []
/content/dataset/kompas []
/content/dataset/mojok []
/content/dataset/etd-ugm []
/content/dataset/tempo []
/content/dataset/etd-usk []


In [32]:
# ============================================================
# PERSIAPAN DATASET IR: ETD-USK, ETD-UGM, KOMPAS, TEMPO, MOJOK
# ============================================================

import os
import re
import json
import gzip
import tarfile
import requests
from bs4 import BeautifulSoup

BASE_DIR = "/content/dataset"
SUMBER = ["etd-usk", "etd-ugm", "kompas", "tempo", "mojok"]

for sumber in SUMBER:
    os.makedirs(os.path.join(BASE_DIR, sumber), exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; AcademicResearch/1.0)"
})

def simpan_dokumen(sumber, nama, isi):
    """Simpan dokumen teks ke folder dataset yang sesuai."""
    isi = re.sub(r"\s+", " ", isi).strip()
    if len(isi) < 100:
        return False

    folder = os.path.join(BASE_DIR, sumber)
    nama = re.sub(r"[^a-zA-Z0-9_-]", "_", nama)[:80]
    path = os.path.join(folder, nama + ".txt")

    with open(path, "w", encoding="utf-8") as f:
        f.write(isi)

    return True


# ------------------------------------------------------------
# 1. KOMPAS DAN TEMPO
# Dataset berita Indonesia 2018, arsip sekitar 446 MB.
# Arsip resmi: indonesian-nlp/id_newspapers_2018
# ------------------------------------------------------------

print("\n[1/5] Mengunduh dataset berita Kompas dan Tempo...")

arsip = "/content/newspapers-json.tgz"
url_arsip = (
    "https://huggingface.co/datasets/"
    "indonesian-nlp/id_newspapers_2018/resolve/main/"
    "newspapers-json.tgz"
)

try:
    if not os.path.exists(arsip):
        with session.get(url_arsip, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(arsip, "wb") as f:
                for bagian in r.iter_content(chunk_size=1024 * 1024):
                    if bagian:
                        f.write(bagian)

    print("Unduhan selesai. Membaca arsip...")

    # Baca arsip tanpa mengekstrak semua file ke disk.
    jumlah = {"kompas": 0, "tempo": 0}
    batas_per_sumber = 100

    with tarfile.open(arsip, "r:gz") as tar:
        for anggota in tar:
            if not anggota.isfile():
                continue

            nama_anggota = anggota.name.lower()

            # Hanya ambil artikel yang nama/path-nya menunjukkan
            # sumber Kompas atau Tempo.
            if "kompas" in nama_anggota:
                sumber = "kompas"
            elif "tempo" in nama_anggota:
                sumber = "tempo"
            else:
                continue

            if jumlah[sumber] >= batas_per_sumber:
                continue

            file_obj = tar.extractfile(anggota)
            if file_obj is None:
                continue

            try:
                mentah = file_obj.read().decode("utf-8", errors="ignore")
                try:
                    data = json.loads(mentah)
                    if isinstance(data, dict):
                        isi = " ".join(
                            str(v) for v in data.values()
                            if isinstance(v, (str, int, float))
                        )
                    else:
                        isi = str(data)
                except json.JSONDecodeError:
                    isi = mentah

                if simpan_dokumen(
                    sumber, f"berita_{jumlah[sumber]+1}", isi
                ):
                    jumlah[sumber] += 1

            except Exception:
                continue

    print("Berita berhasil disimpan:", jumlah)

except Exception as e:
    print("Dataset berita gagal diunduh/dibaca:", e)
    print("Folder berita masih kosong; lanjut mencoba sumber lain.")


# ------------------------------------------------------------
# 2. ETD USK
# Ambil judul dan abstrak dari halaman pencarian publik.
# ------------------------------------------------------------

print("\n[2/5] Mengambil data ETD USK...")

try:
    url = "https://etd.usk.ac.id/search/"
    response = session.get(url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    teks = soup.get_text(" ", strip=True)

    # Simpan halaman katalog sebagai dokumen awal jika berisi teks.
    if simpan_dokumen("etd-usk", "katalog_usk", teks):
        print("Dokumen katalog ETD USK berhasil disimpan.")
    else:
        print("Teks katalog ETD USK terlalu pendek.")

except Exception as e:
    print("ETD USK gagal diakses:", e)


# ------------------------------------------------------------
# 3. ETD UGM
# Ambil teks katalog publik yang bisa diakses tanpa login.
# ------------------------------------------------------------

print("\n[3/5] Mengambil data ETD UGM...")

try:
    url = "https://etd.repository.ugm.ac.id/home"
    response = session.get(url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    teks = soup.get_text(" ", strip=True)

    if simpan_dokumen("etd-ugm", "katalog_ugm", teks):
        print("Dokumen katalog ETD UGM berhasil disimpan.")
    else:
        print("Teks katalog ETD UGM terlalu pendek.")

except Exception as e:
    print("ETD UGM gagal diakses:", e)


# ------------------------------------------------------------
# 4. MOJOK
# Coba mengambil artikel dari halaman publik Mojok.
# ------------------------------------------------------------

print("\n[4/5] Mencoba mengambil artikel Mojok...")

try:
    url = "https://mojok.co/"
    response = session.get(url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    tautan = []

    for a in soup.select("a[href]"):
        href = a.get("href", "")
        judul = a.get_text(" ", strip=True)

        if (
            href.startswith("https://mojok.co/")
            and len(judul) > 20
            and href.rstrip("/") != "https://mojok.co"
            and href not in tautan
        ):
            tautan.append(href)

    jumlah = 0

    for i, link in enumerate(tautan):
        if jumlah >= 10:
            break

        try:
            halaman = session.get(link, timeout=20)
            if halaman.status_code != 200:
                continue

            artikel = BeautifulSoup(halaman.text, "html.parser")

            for tag in artikel(
                ["script", "style", "nav", "footer", "header"]
            ):
                tag.decompose()

            isi = " ".join(
                p.get_text(" ", strip=True)
                for p in artikel.select("article p, main p")
            )

            if simpan_dokumen("mojok", f"artikel_{i+1}", isi):
                jumlah += 1

        except Exception:
            continue

    print(f"Artikel Mojok berhasil disimpan: {jumlah}")

except Exception as e:
    print("Mojok gagal diakses:", e)


# ------------------------------------------------------------
# 5. PERIKSA HASIL
# ------------------------------------------------------------

print("\n[5/5] HASIL DATASET")

total = 0

for sumber in SUMBER:
    folder = os.path.join(BASE_DIR, sumber)
    file_txt = [
        f for f in os.listdir(folder)
        if f.endswith(".txt")
    ]

    print(f"{sumber}: {len(file_txt)} dokumen")
    total += len(file_txt)

print(f"\nTOTAL: {total} dokumen")
print("Lokasi dataset:", BASE_DIR)

if total == 0:
    print(
        "\nBelum ada dokumen yang berhasil diambil. "
        "Jangan jalankan indexing dulu."
    )
else:
    print(
        "\nPeriksa jumlah dokumen setiap sumber. "
        "Katalog ETD saja belum sama dengan kumpulan abstrak tesis."
    )


[1/5] Mengunduh dataset berita Kompas dan Tempo...
Unduhan selesai. Membaca arsip...
Berita berhasil disimpan: {'kompas': 100, 'tempo': 100}

[2/5] Mengambil data ETD USK...
Dokumen katalog ETD USK berhasil disimpan.

[3/5] Mengambil data ETD UGM...
Dokumen katalog ETD UGM berhasil disimpan.

[4/5] Mencoba mengambil artikel Mojok...
Artikel Mojok berhasil disimpan: 0

[5/5] HASIL DATASET
etd-usk: 1 dokumen
etd-ugm: 1 dokumen
kompas: 100 dokumen
tempo: 100 dokumen
mojok: 0 dokumen

TOTAL: 202 dokumen
Lokasi dataset: /content/dataset

Periksa jumlah dokumen setiap sumber. Katalog ETD saja belum sama dengan kumpulan abstrak tesis.


In [33]:
import requests
from bs4 import BeautifulSoup
import os
import re

folder = "/content/dataset/mojok"
os.makedirs(folder, exist_ok=True)

url = "https://mojok.co/"
response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

soup = BeautifulSoup(response.text, "html.parser")

tautan = []
for a in soup.select("a[href]"):
    link = a.get("href", "")
    if link.startswith("https://mojok.co/") and link not in tautan:
        tautan.append(link)

jumlah = 0

for link in tautan:
    if jumlah >= 20:
        break

    try:
        halaman = requests.get(
            link,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=20
        )
        artikel = BeautifulSoup(halaman.text, "html.parser")

        isi = " ".join(
            p.get_text(" ", strip=True)
            for p in artikel.select("article p")
        )

        if len(isi) > 200:
            with open(
                os.path.join(folder, f"mojok_{jumlah+1}.txt"),
                "w",
                encoding="utf-8"
            ) as f:
                f.write(isi)

            jumlah += 1

    except Exception:
        continue

print(f"Artikel Mojok berhasil disimpan: {jumlah}")

Artikel Mojok berhasil disimpan: 3


In [34]:
import os

BASE_DIR = "/content/dataset"
sumber = ["etd-usk", "etd-ugm", "kompas", "tempo", "mojok"]

for nama in sumber:
    folder = os.path.join(BASE_DIR, nama)
    files = [f for f in os.listdir(folder) if f.endswith(".txt")]

    print(f"\n{nama.upper()}: {len(files)} dokumen")

    for file in files[:3]:
        path = os.path.join(folder, file)
        with open(path, "r", encoding="utf-8") as f:
            isi = f.read()

        print(f"- {file}: {len(isi)} karakter")


ETD-USK: 1 dokumen
- katalog_usk.txt: 17001 karakter

ETD-UGM: 1 dokumen
- katalog_ugm.txt: 1279 karakter

KOMPAS: 100 dokumen
- berita_34.txt: 1913 karakter
- berita_52.txt: 2516 karakter
- berita_98.txt: 1394 karakter

TEMPO: 100 dokumen
- berita_34.txt: 1899 karakter
- berita_52.txt: 2898 karakter
- berita_98.txt: 1956 karakter

MOJOK: 3 dokumen
- mojok_2.txt: 329 karakter
- mojok_1.txt: 4332 karakter
- mojok_3.txt: 1566 karakter


In [35]:
from bs4 import BeautifulSoup
import requests

url = "https://etd.usk.ac.id/search/"
response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

soup = BeautifulSoup(response.text, "html.parser")

for a in soup.find_all("a", href=True):
    teks = a.get_text(" ", strip=True)
    link = a["href"]

    if teks and ("detail" in link.lower() or "id=" in link.lower()):
        print(teks[:100], "->", link)

Detail Selengkapnya -> ?p=show_detail&id=1729059
Detail Selengkapnya -> ?p=show_detail&id=1729047
Detail Selengkapnya -> ?p=show_detail&id=1729045
Detail Selengkapnya -> ?p=show_detail&id=1729043
Detail Selengkapnya -> ?p=show_detail&id=1729041
Detail Selengkapnya -> ?p=show_detail&id=1729039
Detail Selengkapnya -> ?p=show_detail&id=1729035
Detail Selengkapnya -> ?p=show_detail&id=1729033
Detail Selengkapnya -> ?p=show_detail&id=1729029
Detail Selengkapnya -> ?p=show_detail&id=1729027
Detail Selengkapnya -> ?p=show_detail&id=1729023
Detail Selengkapnya -> ?p=show_detail&id=1729021
Detail Selengkapnya -> ?p=show_detail&id=1729019
Detail Selengkapnya -> ?p=show_detail&id=1729015
Detail Selengkapnya -> ?p=show_detail&id=1729013


In [36]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_DIR = "/content/dataset/etd-usk"
os.makedirs(BASE_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

# Ambil daftar halaman detail dari katalog ETD-USK
url_katalog = "https://etd.usk.ac.id/search/"
response = session.get(url_katalog, timeout=30)
soup = BeautifulSoup(response.text, "html.parser")

tautan_detail = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    if "show_detail" in href and href not in tautan_detail:
        tautan_detail.append(urljoin(url_katalog, href))

jumlah = 0

for link in tautan_detail[:20]:
    try:
        response = session.get(link, timeout=30)
        halaman = BeautifulSoup(response.text, "html.parser")

        # Ambil teks utama halaman
        isi = halaman.get_text(" ", strip=True)

        if len(isi) > 500:
            nama = f"etd_usk_{jumlah + 1}.txt"
            path = os.path.join(BASE_DIR, nama)

            with open(path, "w", encoding="utf-8") as f:
                f.write(isi)

            jumlah += 1

    except Exception as e:
        print("Gagal mengambil halaman:", e)

print(f"Berhasil menyimpan {jumlah} dokumen ETD-USK.")

Berhasil menyimpan 15 dokumen ETD-USK.


In [37]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_DIR = "/content/dataset/etd-ugm"
os.makedirs(BASE_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

url = "https://etd.repository.ugm.ac.id/home"
response = session.get(url, timeout=30)
soup = BeautifulSoup(response.text, "html.parser")

tautan = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    if any(k in href.lower() for k in ["detail", "id="]):
        link = urljoin(url, href)
        if link not in tautan:
            tautan.append(link)

jumlah = 0

for link in tautan[:20]:
    try:
        response = session.get(link, timeout=30)
        halaman = BeautifulSoup(response.text, "html.parser")
        isi = halaman.get_text(" ", strip=True)

        if len(isi) > 500:
            path = os.path.join(BASE_DIR, f"etd_ugm_{jumlah+1}.txt")
            with open(path, "w", encoding="utf-8") as f:
                f.write(isi)
            jumlah += 1

    except Exception as e:
        print("Gagal mengambil halaman:", e)

print(f"Berhasil menyimpan {jumlah} dokumen ETD-UGM.")

Berhasil menyimpan 15 dokumen ETD-UGM.


In [38]:
import os

BASE_DIR = "/content/dataset"
sumber = ["etd-usk", "etd-ugm", "kompas", "tempo", "mojok"]

total = 0

for nama in sumber:
    folder = os.path.join(BASE_DIR, nama)
    files = [
        f for f in os.listdir(folder)
        if f.endswith(".txt")
    ]
    print(f"{nama}: {len(files)} dokumen")
    total += len(files)

print(f"\nTOTAL DOKUMEN: {total}")

etd-usk: 16 dokumen
etd-ugm: 16 dokumen
kompas: 100 dokumen
tempo: 100 dokumen
mojok: 3 dokumen

TOTAL DOKUMEN: 235


In [23]:
%pip install -q Whoosh scikit-learn